# Train Film Index
Create a Kaggle Notebook, import this file, and add the [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) as an input. Select a CPU session and run all cells. Download `catalog.pkl` from the notebook output and place it in the app’s `artifacts/` directory.

The cells below use the app’s training and validation functions. Kaggle’s preinstalled Python 3.11+ scientific stack is sufficient. The project’s Python 3.12 environment and pinned dependencies reproduce the bundled release.

In [ ]:
import hashlib
import json
import math
import os
import pickle
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

input_root = Path(os.environ.get("FILMINDEX_DATA_ROOT", "/kaggle/input"))
pairs = [p.parent for p in input_root.rglob("tmdb_5000_movies.csv")
         if (p.parent / "tmdb_5000_credits.csv").is_file()]
if len(pairs) != 1:
    raise ValueError("Attach one matching TMDB 5000 dataset or set FILMINDEX_DATA_ROOT to its folder.")
data_dir = pairs[0]
output = Path(os.environ.get("FILMINDEX_OUTPUT", "/kaggle/working/catalog.pkl"))
print("Dataset:", data_dir, "| scikit-learn:", sklearn.__version__)

## Verify the Kaggle snapshot

In [ ]:
FILES = {
    "tmdb_5000_movies.csv": "1e0584a8dd374120e4ed4f2b81f0d2bd0eb95a554708e4a350d00ed5c46d2825",
    "tmdb_5000_credits.csv": "9d0050599ff88d40366c4841204b1489862bca346bfa46c20b05a65d14508435",
}

def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

for name, expected in FILES.items():
    if file_hash(data_dir / name) != expected:
        raise ValueError(f"Unexpected Kaggle snapshot for {name}")

## Join films to credits and build sparse TF–IDF features
Each metadata field gets its own vocabulary. Actor names and multiword keywords stay intact.

In [ ]:
WEIGHTS = {"overview": 0.24, "genres": 0.20, "keywords": 0.36, "cast": 0.12, "directors": 0.08}

def names(raw, field, movie_id, *, job=None, limit=None):
    if not raw:
        return []
    try:
        entries = json.loads(raw)
        if not isinstance(entries, list):
            raise ValueError("expected a list")
        if job:
            entries = [e for e in entries if e.get("job") == job]
        if field == "cast":
            entries.sort(key=lambda e: e.get("order", len(entries)))
        cleaned = [e["name"].strip() for e in entries if e.get("name")]
        return list(dict.fromkeys(cleaned))[:limit]
    except (ValueError, TypeError, KeyError, AttributeError) as exc:
        raise ValueError(f"Invalid {field} metadata for TMDB ID {movie_id}") from exc


def read_movies(directory):
    films = pd.read_csv(directory / "tmdb_5000_movies.csv", keep_default_na=False,
                        usecols=["id", "title", "release_date", "overview", "genres", "keywords"])
    credits = pd.read_csv(directory / "tmdb_5000_credits.csv", keep_default_na=False,
                          usecols=["movie_id", "cast", "crew"]).rename(columns={"movie_id": "id"})
    merged = films.merge(credits, on="id", how="outer", validate="one_to_one", indicator=True)
    if not (merged["_merge"] == "both").all():
        raise ValueError("Movie and credits IDs differ. Use a matching dataset pair.")
    movies = {}
    for row in merged.sort_values("id").itertuples(index=False):
        if not row.title.strip() or row.id <= 0:
            raise ValueError(f"Invalid movie identity: {row.id}")
        movies[int(row.id)] = {
            "title": row.title.strip(), "year": row.release_date[:4], "overview": row.overview.strip(),
            "genres": names(row.genres, "genres", row.id),
            "keywords": names(row.keywords, "keywords", row.id),
            "cast": names(row.cast, "cast", row.id, limit=5),
            "directors": names(row.crew, "crew", row.id, job="Director"),
        }
    return movies


def identity(tokens):
    return tokens


def features(movies):
    blocks = []
    for field, weight in WEIGHTS.items():
        if field == "overview":
            corpus = [m[field] for m in movies.values()]
            vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True, strip_accents="unicode",
                                         dtype=np.float32, max_features=40000, ngram_range=(1, 2))
        else:
            corpus = [[re.sub(r"\s+", " ", term.casefold()) for term in m[field]] for m in movies.values()]
            vectorizer = TfidfVectorizer(analyzer=identity, dtype=np.float32)
        try:
            block = vectorizer.fit_transform(corpus)
        except ValueError as exc:
            if "empty vocabulary" in str(exc):
                continue
            raise
        blocks.append(block.multiply(np.float32(weight ** 0.5)))
    if not blocks:
        raise ValueError("No usable content features")
    matrix = normalize(hstack(blocks, format="csr"))
    if np.any(matrix.getnnz(axis=1) == 0):
        raise ValueError("Some movies have no usable content. Check the input metadata.")
    return matrix

## Compute the five nearest neighbors
Each batch compares up to 128 films against the catalog. A movie cannot recommend itself. Ties resolve by ascending TMDB ID.

In [ ]:
TOP_K = 5
SCHEMA = 1

def nearest(movies, matrix, batch_size=128):
    ids = np.array(list(movies))
    if len(ids) <= TOP_K:
        raise ValueError("At least six movies are required")
    result = {}
    for start in range(0, len(ids), batch_size):
        scores = cosine_similarity(matrix[start:start + batch_size], matrix)
        for offset, row in enumerate(scores):
            index = start + offset
            row[index] = -np.inf
            # IDs are ascending, providing a stable tie break at the top-five boundary.
            best = np.argsort(-row, kind="stable")[:TOP_K]
            result[int(ids[index])] = [(int(ids[j]), float(np.clip(row[j], 0, 1))) for j in best]
    return result

## Validate and save the serving index

In [ ]:
class CatalogError(ValueError):
    """The deployed release is missing, damaged, or unsupported."""


class Catalog:
    def __init__(self, release):
        if release["schema"] != SCHEMA:
            raise CatalogError("Unsupported catalog version")
        self.movies = release["movies"]
        self.neighbors = release["neighbors"]
        self.metadata = release["metadata"]
        if len(self.movies) <= TOP_K or self.movies.keys() != self.neighbors.keys():
            raise CatalogError("Incomplete catalog")
        for movie_id, movie in self.movies.items():
            if not isinstance(movie_id, int) or not movie["title"]:
                raise CatalogError("Invalid movie record")
            for key in ("genres", "directors", "keywords", "cast", "overview", "year", "title"):
                if key not in movie:
                    raise CatalogError("Incomplete movie record")
            matches = self.neighbors[movie_id]
            ids = {other for other, _ in matches}
            if len(matches) != TOP_K or len(ids) != TOP_K or movie_id in ids or not ids <= self.movies.keys():
                raise CatalogError("Invalid recommendation IDs")
            scores = [score for _, score in matches]
            if any(not math.isfinite(s) or not 0 <= s <= 1 for s in scores) or scores != sorted(scores, reverse=True):
                raise CatalogError("Invalid cosine scores")
        self.sorted_ids = sorted(self.movies, key=lambda i: (self.movies[i]["title"].casefold(), self.movies[i]["year"], i))

    def label(self, movie_id):
        movie = self.movies[movie_id]
        return f"{movie['title']} ({movie['year'] or 'year unknown'})"

    def recommend(self, movie_id):
        return self.neighbors[movie_id]


def train(data_dir, output):
    hashes = {name: file_hash(data_dir / name) for name in FILES}
    movies = read_movies(data_dir)
    matrix = features(movies)
    release = {"schema": SCHEMA, "movies": movies, "neighbors": nearest(movies, matrix), "metadata": {
        "weights": WEIGHTS, "input_sha256": hashes, "sklearn": sklearn.__version__,
        "movie_count": len(movies), "feature_count": matrix.shape[1],
    }}
    Catalog(release)
    output.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=output.parent, delete=False) as temp:
        path = Path(temp.name)
        try:
            pickle.dump(release, temp, protocol=5)
            temp.flush()
            os.fsync(temp.fileno())
            temp.close()
            os.replace(path, output)
        finally:
            path.unlink(missing_ok=True)
    print(f"Saved {len(movies):,} films and {matrix.shape[1]:,} features to {output}")

train(data_dir, output)

## Check the result
The release contains movie metadata, five IDs and scores per movie, feature weights, the scikit-learn version and the source file hashes.

In [ ]:
class _CatalogReader(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError("Catalogs cannot import Python objects")

with output.open("rb") as stream:
    catalog = Catalog(_CatalogReader(stream).load())
for movie_id, score in catalog.recommend(862):
    print(f"{catalog.label(movie_id):45} {score:.3f}")
print(f"Download {output} and place it under artifacts/ in your Streamlit project.")